# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 11_12_15_main_geometry_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-03
### Geometry generation script
**Goal:** Generate the geometry dataset used as input for the Grasshopper structural analysis. The script accepts several variables and writes a CSV file containing the properties needed to reconstruct the digital geometry as CAD geometry.

**Inputs:**   
*   Base mesh definition
*   Allowed movement for different vertices
*   Divisions over the allowed movement

**Outputs:**

*   CSV file with sample ID, vertices, normalized coordinates, and edges
*   Centroid and PCA alignment are applied once in generation; do not repeat that normalization in Grasshopper

### Testing corners

In [1]:
import c00_headquarter_params as c11_params
import config
from c12_geometry_truss import get_corner_indices

# --- TEST ---
grid_2x2 = get_corner_indices(2, 2)
print(f"2x2 grid indices: {grid_2x2}")

grid_3x3 = get_corner_indices(3, 3)
print(f"3x3 grid indices: {grid_3x3}")

grid_5x3 = get_corner_indices(5, 3)
print(f"5x3 grid indices: {grid_5x3}")
# Expected: 0 (BL), 4 (BR), 20 (TL), 24 (TR) -> because there are 5 points per row

# In your loop:
corners = get_corner_indices(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)
corner_values = corners.values() # [0, 4, 20, 24] for example
print(corner_values)

GRID: 5x3, EDGE_LENGTH: 3.0, LAYER_HEIGHT: 1.5, DIVISIONS: 8, NUM_SAMPLES: 20000

IMPACT_FACTOR_A1_A3: 0.25, IMPACT_FACTOR_RECOVERED_C1: 0.0085, "ENERGY_PREP_SAW_A5: 0.02, ENERGY_OFFCUT_FACTOR_C3_C4: 0.276, SCARCITY_PENALTY: 0

parameters loaded from c:\Users\jaspe\Documents\PyRepo\thesis_generative_timber\c00_headquarter_params.py
Config System loaded successfully, Code running locally from thesis_generative_timber and Data is connected to OneDrive 2.2 - 2.4.

2x2 grid indices: {'bottom_left': 0, 'bottom_right': 2, 'top_left': 6, 'top_right': 8}
3x3 grid indices: {'bottom_left': 0, 'bottom_right': 3, 'top_left': 12, 'top_right': 15}
5x3 grid indices: {'bottom_left': 0, 'bottom_right': 5, 'top_left': 18, 'top_right': 23}
dict_values([0, 5, 18, 23])


## 1.2 Executing and saving dataset

In [2]:
import pandas as pd
import numpy as np
import config
import c00_headquarter_params as c11_params
from c12_geometry_truss import generate_vertices, generate_edges

# --- RUN ---
final_vertices = generate_vertices(c11_params.NUM_SAMPLES)
final_edges = generate_edges(c11_params.NUM_SAMPLES, c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)

# --- SAVE ---
final_vertices_name = f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_vertices.csv"
final_vertices.to_csv(config.GEOM_DATA_PATH / final_vertices_name, index=False)
final_edges.to_csv(config.GEOM_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_edges.csv", index=False)

print(f"Generation complete for {c11_params.NUM_SAMPLES} samples.")
print(f"Grid size: {c11_params.GRID}")

print("Vertices shape:", final_vertices.shape)
print("Edges shape:", final_edges.shape)

print(f"\nTotal rows in vertices CSV: {len(final_vertices)}")
print("\nExample output (first 5 rows of sample 0):")
print(final_vertices.head(5))
print("\nExample output (first 5 rows of sample 1):")
print(final_vertices[final_vertices['sample_id'] == 1].head(5))

print(f"\nTotal rows in edges CSV: {len(final_edges)}")
print("\nExample output (first 5 rows of sample 0):")
print(final_edges.head(5))
print("\nExample output (first 5 rows of sample 1):")
print(final_edges[final_edges['sample_id'] == 1].head(5))

Generation complete for 20000 samples.
Grid size: 5x3
Vertices shape: (780000, 7)
Edges shape: (2400000, 4)

Total rows in vertices CSV: 780000

Example output (first 5 rows of sample 0):
   sample_id vertex_index layer attribute      x      y      z
0          0           v0   top   support -7.539 -4.276  0.462
1          0           v1   top      load -3.789 -4.276  0.462
2          0           v2   top      load -1.164 -4.276  0.462
3          0           v3   top      load  1.836 -4.276  0.462
4          0           v4   top      load  4.461 -4.276  0.462

Example output (first 5 rows of sample 1):
    sample_id vertex_index layer attribute      x      y      z
39          1           v0   top   support -7.938 -4.646  0.558
40          1           v1   top      load -3.813 -4.646  0.558
41          1           v2   top      load -1.563 -4.646  0.558
42          1           v3   top      load  1.437 -4.646  0.558
43          1           v4   top      load  4.437 -4.646  0.558

Total

## 1.5 DESIGN DOMAIN
two output from the geometry script need to be transferred to json files for further use in the script, these are:
*   Search Space, this is the space where the vertices can move, this is nessecary for the optimizing of the geometry so the optimizer knows where it can and cant go.
*   Edge Index, the edge index is nessecary for the training of the surrogate model, this is used because the relationship of the vertices is the same vor every structure, also lowers this the number of features (x) used in training for a better generalisation.

In [3]:
import json
import config
import c00_headquarter_params as c11_params
from c12_geometry_truss import generate_edges, define_search_space

# --- RUN AND REVIEW ---
# Use the variables from your earlier configuration
search_space = define_search_space(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y, c11_params.DIVISIONS, c11_params.EDGE_LENGTH)
df_topology = generate_edges(num_samples=1, cells_x=c11_params.GRID_CELLS_X, cells_y=c11_params.GRID_CELLS_Y)
edge_index = df_topology[['V1', 'V2']].values.T.tolist()

print(f"The search space contains {len(search_space)} independent variables (the controls the AI can adjust).")
print(f"Topology generated. Number of unique connections (edges): {len(df_topology)}")
print("\nExample of how the computer sees this:")

# Show the first 5 variables in a readable format
for var_name, bounds in list(search_space.items())[:5]:
    print(f"- {var_name}: {bounds}")

print("\nGenerated edge_index (first 5 connections):")
print(f"Source nodes (V1): {edge_index[0][:5]}")
print(f"Target nodes (V2): {edge_index[1][:5]}")

# Save the dictionaries as JSON files
json_search_space_path = config.DATA_IO_PATH / "search_space.json"
json_edge_index_path = config.DATA_IO_PATH / "edge_index.json"
with open(json_search_space_path, 'w') as f:
    json.dump(search_space, f, indent=4) # indent=4 makes it easy to read
with open(json_edge_index_path, 'w') as f:
    json.dump(edge_index, f)

print(f"Search space successfully saved as '{json_search_space_path}'")
print(f"\nEdge index (as a plain list) successfully saved to '{json_edge_index_path}'.")

The search space contains 73 independent variables (the controls the AI can adjust).
Topology generated. Number of unique connections (edges): 120

Example of how the computer sees this:
- v1_shift_x: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v2_shift_x: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v3_shift_x: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v4_shift_x: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v6_shift_y: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}

Generated edge_index (first 5 connections):
Source nodes (V1): [0, 0, 1, 1, 2]
Target nodes (V2): [1, 6, 2, 7, 3]
Search space successfully saved as 'C:\Users\jaspe\Documents\PyRepo\thesis_generative_timber\02_data_io\search_space.json'

Edge index (as a plain list) successfully saved to 'C:\Users\jaspe\Documents\PyRepo\thesis_gen

## Average Length
Here the average length of the geometry is calculated for the generation of the timber stock

In [ ]:
import config
import json
import importlib
import pandas as pd
import c00_headquarter_params as c11_params
import c12_reconstruction

importlib.reload(c12_reconstruction)

# Load vertices and edge index from disk so this cell can run independently
final_vertices_path = config.GEOM_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_vertices.csv"
with open(config.DATA_IO_PATH / "edge_index.json", 'r', encoding='utf-8') as f:
    edge_index = json.load(f)

final_vertices = pd.read_csv(final_vertices_path)

# Generate material statistics using the loaded vertices dataframe
result = c12_reconstruction.generate_material_statistics(
    df_vertices=final_vertices,
    edge_index=edge_index,
    sample_count=10,
    random_state=42
)

summary = result["summary_statistics"]

print("Representative statistics:")
print(f"  Representative length: {result['representative_length_m']:.3f}m ({result['representative_length_mm']:.1f}mm)")
print(f"  Average length: {summary['average_length_m']:.3f}m ({summary['average_length_mm']:.1f}mm)")
print(f"  Median length: {summary['median_length_m']:.3f}m ({summary['median_length_mm']:.1f}mm)")
print(f"  Min length: {summary['min_length_m']:.3f}m ({summary['min_length_mm']:.1f}mm)")
print(f"  Max length: {summary['max_length_m']:.3f}m ({summary['max_length_mm']:.1f}mm)")
print(f"  Total length: {summary['total_length_m']:.3f}m ({summary['total_length_mm']:.1f}mm)")
print(f"  Std dev: {summary['std_dev_m']:.3f}m ({summary['std_dev_mm']:.1f}mm)")
print(f"  Q1: {summary['q1_m']:.3f}m ({summary['q1_mm']:.1f}mm)")
print(f"  Q3: {summary['q3_m']:.3f}m ({summary['q3_mm']:.1f}mm)")
print(f"  Average edge count: {summary['edge_count']}")
print(f"Selected samples: {result['selected_sample_ids']}\n")

print("Per-sample statistics:")
for sample_result in result['per_sample_results']:
    print(f"\nSample {sample_result['sample_id']}:")
    print(f"  Average: {sample_result['average_length_m']:.3f}m ({sample_result['average_length_mm']:.1f}mm)")
    print(f"  Median: {sample_result['median_length_m']:.3f}m ({sample_result['median_length_mm']:.1f}mm)")
    print(f"  Min: {sample_result['min_length_m']:.3f}m ({sample_result['min_length_mm']:.1f}mm)")
    print(f"  Max: {sample_result['max_length_m']:.3f}m ({sample_result['max_length_mm']:.1f}mm)")
    print(f"  Total: {sample_result['total_length_m']:.3f}m ({sample_result['total_length_mm']:.1f}mm, {sample_result['edge_count']} beams)")

# Save to JSON
with open(config.DATA_IO_PATH / "representative_beam_statistics.json", 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2)

Representative statistics:
  Representative length: 2.979m (2979.0mm)
  Average length: 2.978m (2978.3mm)
  Median length: 2.975m (2975.3mm)
  Min length: 1.101m (1101.4mm)
  Max length: 5.102m (5102.2mm)
  Total length: 357.400m (357399.7mm)
  Std dev: 0.804m (803.8mm)
  Q1: 2.388m (2388.0mm)
  Q3: 3.515m (3514.7mm)
  Average edge count: 120
Selected samples: [3648, 819, 9012, 8024, 7314, 4572, 3358, 17870, 2848, 19349]

Per-sample statistics:

Sample 3648:
  Average: 2.885m (2884.9mm)
  Median: 2.897m (2896.9mm)
  Min: 1.264m (1264.4mm)
  Max: 4.875m (4875.0mm)
  Total: 346.188m (346188.5mm, 120 beams)

Sample 819:
  Average: 3.038m (3038.1mm)
  Median: 3.062m (3061.8mm)
  Min: 1.186m (1185.9mm)
  Max: 4.875m (4875.0mm)
  Total: 364.572m (364571.7mm, 120 beams)

Sample 9012:
  Average: 3.027m (3027.3mm)
  Median: 3.000m (3000.0mm)
  Min: 1.125m (1125.0mm)
  Max: 5.263m (5263.4mm)
  Total: 363.272m (363272.4mm, 120 beams)

Sample 8024:
  Average: 2.973m (2972.9mm)
  Median: 2.994m (29